# pyCAFE — Modal Analysis Example

This notebook walks through a **modal (eigenvalue) acoustic analysis** of a 2D rectangular cavity.

Steps:
1. Create the mesh with Gmsh
2. Load the mesh and inspect boundary names
3. **Select boundary conditions interactively** (one dropdown per boundary)
4. Assemble the FEM system
5. Solve the eigenvalue problem
6. Visualise mode shapes (matplotlib)
7. Visualise mode shapes inline with **pyvista**
8. *(Optional)* Export results to VTK / ParaView

---
**Geometry:** rectangular 2D cavity  
**Element type:** CQUAD8 (serendipity quadrilateral, order 2)  
**Fluid:** air at 20 °C

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
import tempfile, pathlib

import gmsh
import pycafe
from pycafe.solver.solver_modale import solve_modal_acoustic_reduced
from pycafe.build_matrices.assembly_cquad8 import expand_mode_to_full

## Step 1 — Geometry and mesh parameters

Edit the cell below to change the cavity size, mesh density, and fluid properties.

In [ ]:
# ── Cavity dimensions ─────────────────────────────────────────────────────────
Lx = 1.0    # length [m]
Ly = 0.5    # height [m]

# ── Fluid properties ──────────────────────────────────────────────────────────
rho = 1.204   # density [kg/m³]
c0  = 343.0   # speed of sound [m/s]

# ── Mesh ─────────────────────────────────────────────────────────────────────
h     = 0.07  # target element size [m]
order = 2     # 1 = CQUAD4, 2 = CQUAD8

# ── Modal solver ─────────────────────────────────────────────────────────────
N_MODES = 8   # number of acoustic modes to compute

## Step 2 — Generate and load the mesh

In [ ]:
def make_rect_mesh(Lx, Ly, h, order, filepath):
    """Structured transfinite quad mesh for a rectangle."""
    try:
        if gmsh.isInitialized():
            gmsh.finalize()
    except Exception:
        pass
    gmsh.initialize()
    gmsh.option.setNumber("General.Verbosity", 0)
    gmsh.model.add("cavity")
    p1 = gmsh.model.geo.addPoint(0,  0,  0, h)
    p2 = gmsh.model.geo.addPoint(Lx, 0,  0, h)
    p3 = gmsh.model.geo.addPoint(Lx, Ly, 0, h)
    p4 = gmsh.model.geo.addPoint(0,  Ly, 0, h)
    l1 = gmsh.model.geo.addLine(p1, p2)
    l2 = gmsh.model.geo.addLine(p2, p3)
    l3 = gmsh.model.geo.addLine(p3, p4)
    l4 = gmsh.model.geo.addLine(p4, p1)
    cl   = gmsh.model.geo.addCurveLoop([l1, l2, l3, l4])
    surf = gmsh.model.geo.addPlaneSurface([cl])
    gmsh.model.addPhysicalGroup(1, [l1], name="bottom")
    gmsh.model.addPhysicalGroup(1, [l2], name="right")
    gmsh.model.addPhysicalGroup(1, [l3], name="top")
    gmsh.model.addPhysicalGroup(1, [l4], name="left")
    gmsh.model.addPhysicalGroup(2, [surf], name="domain")
    gmsh.model.geo.synchronize()
    nx = max(int(round(Lx / h)) + 1, 3)
    ny = max(int(round(Ly / h)) + 1, 3)
    for line in [l1, l3]:
        gmsh.model.mesh.setTransfiniteCurve(line, nx)
    for line in [l2, l4]:
        gmsh.model.mesh.setTransfiniteCurve(line, ny)
    gmsh.model.mesh.setTransfiniteSurface(surf)
    gmsh.option.setNumber("Mesh.SecondOrderIncomplete", 1)
    gmsh.model.mesh.generate(2)
    gmsh.model.mesh.recombine()
    gmsh.model.mesh.setOrder(order)
    gmsh.write(str(filepath))
    gmsh.finalize()

_msh_file = pathlib.Path(tempfile.mktemp(suffix=".msh"))
make_rect_mesh(Lx, Ly, h, order, _msh_file)
nodes, elements, boundaries = pycafe.load_mesh(str(_msh_file), show_info=True, show_plot=True)
_msh_file.unlink(missing_ok=True)

print(f"\nMesh loaded: {nodes.shape[0]} nodes")
print(f"Boundaries available: {list(boundaries.keys())}")

## Step 3 — Select boundary conditions

Use the dropdown menus below to assign a boundary condition to each wall.

| Option | Description |
|--------|-------------|
| **Hard wall** | Rigid wall — zero normal velocity (default Neumann BC) |
| **Zero pressure** | Pressure node constrained to p = 0 Pa |
| **Constant pressure** | Prescribed constant pressure amplitude |
| **Impedance** | Acoustic impedance Z (complex, [Pa·s/m]) |
| **Normal velocity** | Prescribed normal velocity amplitude [m/s] |

> **Note for modal analysis:** Hard wall BCs on all boundaries give the classic *rigid cavity* eigenvalue problem with an analytical solution. Adding a zero-pressure boundary introduces a pressure-release (soft) wall and shifts the eigenfrequencies.

In [ ]:
BC_OPTIONS = [
    "Hard wall (rigid)",
    "Zero pressure",
    "Constant pressure",
    "Impedance",
    "Normal velocity",
]

_bc_dropdowns = {}
_bc_rows = []

for bname in boundaries:
    if bname == "domain":
        continue
    dd = widgets.Dropdown(
        options=BC_OPTIONS,
        value="Hard wall (rigid)",
        description=f"{bname}:",
        style={"description_width": "120px"},
        layout=widgets.Layout(width="380px"),
    )
    _bc_dropdowns[bname] = dd
    _bc_rows.append(dd)

_extra_label = widgets.HTML("<b>Extra parameters (used only for the selected BC types):</b>")
_w_pressure_val = widgets.FloatText(value=1.0,   description="Pressure [Pa]:",
                                     style={"description_width": "140px"}, layout=widgets.Layout(width="280px"))
_w_imp_real     = widgets.FloatText(value=415.0, description="Z real [Pa·s/m]:",
                                     style={"description_width": "140px"}, layout=widgets.Layout(width="280px"))
_w_imp_imag     = widgets.FloatText(value=0.0,   description="Z imag [Pa·s/m]:",
                                     style={"description_width": "140px"}, layout=widgets.Layout(width="280px"))
_w_velocity     = widgets.FloatText(value=1.0,   description="v_n [m/s]:",
                                     style={"description_width": "140px"}, layout=widgets.Layout(width="280px"))

display(
    widgets.VBox(_bc_rows + [
        widgets.HTML("<hr>"),
        _extra_label,
        _w_pressure_val,
        _w_imp_real, _w_imp_imag,
        _w_velocity,
    ])
)

In [ ]:
bc_pressure_zero      = []
bc_pressure_constant  = []
bc_impedance          = []
bc_velocity           = []

for bname, dd in _bc_dropdowns.items():
    choice = dd.value
    if choice == "Zero pressure":
        bc_pressure_zero.append(bname)
    elif choice == "Constant pressure":
        bc_pressure_constant.append(bname)
    elif choice == "Impedance":
        bc_impedance.append(bname)
    elif choice == "Normal velocity":
        bc_velocity.append(bname)

bc = (
    bc_pressure_zero,
    bc_pressure_constant,
    float(_w_pressure_val.value),
    bc_impedance,
    complex(_w_imp_real.value, _w_imp_imag.value),
    bc_velocity,
    float(_w_velocity.value),
    None,
    0.0,
)

print("Boundary conditions summary")
print(f"  Hard wall     : {[b for b,d in _bc_dropdowns.items() if d.value == 'Hard wall (rigid)']}")
print(f"  Zero pressure : {bc_pressure_zero}")
print(f"  Const. pressure ({_w_pressure_val.value} Pa) : {bc_pressure_constant}")
print(f"  Impedance ({_w_imp_real.value}+{_w_imp_imag.value}j Pa·s/m) : {bc_impedance}")
print(f"  Normal velocity ({_w_velocity.value} m/s) : {bc_velocity}")

## Step 4 — Assemble the FEM system

In [ ]:
system = pycafe.prepare_acoustic_system(
    nodes=nodes,
    elements=elements,
    boundaries=boundaries,
    rho=rho,
    c0=c0,
    bc=bc,
    debug=False,
)
print(f"System assembled. Reduced DOFs: {system['K_red'].shape[0]}")

## Step 5 — Solve the eigenvalue problem

In [ ]:
freqs_raw, modes_red = solve_modal_acoustic_reduced(
    system["K_red"],
    system["M_red"],
    num_modes=N_MODES + 4,
)

mask      = freqs_raw > 1.0
freqs     = freqs_raw[mask][:N_MODES]
modes_red = modes_red[:, mask][:, :N_MODES]

print(f"\n{'Mode':>5}  {'Frequency [Hz]':>16}")
print("-" * 25)
for i, f in enumerate(freqs):
    print(f"  {i+1:>3}  {f:>16.4f}")

## Step 6 — Visualise mode shapes (matplotlib)

Use the slider to switch between modes.

In [ ]:
def plot_mode(mode_index):
    k = mode_index - 1
    mode_full = expand_mode_to_full(
        modes_red[:, k], system["idx_free"], system["p0_nodes"], nodes.shape[0]
    )
    mode_plot = np.real(mode_full)
    peak = np.max(np.abs(mode_plot))
    if peak > 0:
        mode_plot /= peak

    fig, ax = plt.subplots(figsize=(8, 4))
    sc = ax.scatter(nodes[:, 0], nodes[:, 1], c=mode_plot,
                    cmap="RdBu_r", s=20, vmin=-1, vmax=1)
    plt.colorbar(sc, ax=ax, label="Normalised amplitude")
    ax.set_aspect("equal")
    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")
    ax.set_title(f"Mode {mode_index} — f = {freqs[k]:.4f} Hz")
    ax.grid(True, linewidth=0.4)
    plt.tight_layout()
    plt.show()

widgets.interact(
    plot_mode,
    mode_index=widgets.IntSlider(
        value=1, min=1, max=len(freqs), step=1,
        description="Mode:",
        style={"description_width": "60px"},
        layout=widgets.Layout(width="400px"),
    ),
);

## Step 7 — Visualise mode shapes inline with pyvista

The cell below builds a `pyvista.UnstructuredGrid` directly from the pyCAFE mesh arrays and renders the selected mode shape inline in the notebook.

- Edges of the elements are drawn to show the mesh topology
- Colormap `RdBu_r` highlights nodal lines (zero crossings) in white
- Use the slider to switch mode; re-run the cell after changing it

In [ ]:
import pyvista as pv

pv.set_jupyter_backend("trame")  # interactive inline rendering

# ── Build pyvista UnstructuredGrid from pyCAFE arrays ────────────────────────
pts3d = nodes.copy().astype(np.float64)
if pts3d.shape[1] == 2:
    pts3d = np.column_stack([pts3d, np.zeros(len(pts3d))])

# locate quad elements (skip line / boundary entities)
_quad_key = next(
    k for k in elements
    if "Quadrilateral" in k or "Quadrangle" in k or "quad" in k.lower()
)
conn       = elements[_quad_key].astype(np.int64) - 1  # 0-based
n_cells, n_per_cell = conn.shape
vtk_type   = 23 if n_per_cell == 8 else 9              # QUAD8 or QUAD4

# pyvista cell array format: [n_pts, id0, id1, ..., n_pts, id0, ...]
_cells_flat = np.hstack(
    [np.full((n_cells, 1), n_per_cell, dtype=np.int64), conn]
).ravel()
_cell_types = np.full(n_cells, vtk_type, dtype=np.uint8)

pv_grid = pv.UnstructuredGrid(_cells_flat, _cell_types, pts3d)
print(f"pyvista grid: {pv_grid.n_points} points, {pv_grid.n_cells} cells, type QUAD{n_per_cell}")

In [ ]:
# ── Choose which mode to display ─────────────────────────────────────────────
MODE_TO_SHOW = 1   # ← change this (1-based)

k = MODE_TO_SHOW - 1
mode_full = expand_mode_to_full(
    modes_red[:, k], system["idx_free"], system["p0_nodes"], nodes.shape[0]
)
mode_plot = np.real(mode_full).astype(np.float64)
peak = np.max(np.abs(mode_plot))
if peak > 0:
    mode_plot /= peak

pv_grid.point_data["mode_shape"] = mode_plot

plotter = pv.Plotter(notebook=True)
plotter.add_mesh(
    pv_grid,
    scalars="mode_shape",
    cmap="RdBu_r",
    clim=[-1.0, 1.0],
    show_edges=True,
    edge_color="grey",
    scalar_bar_args={"title": "Normalised\namplitude", "n_labels": 5},
)
plotter.add_title(f"Mode {MODE_TO_SHOW}  —  f = {freqs[k]:.4f} Hz", font_size=12)
plotter.view_xy()
plotter.show()

## Step 8 — (Optional) Export to VTK / ParaView

Run the cell below to write all mode shapes as `.vtu` files and a `.pvd` collection.  
Open `vtu_modes/modal_results.pvd` in ParaView to animate through modes.

In [ ]:
from pycafe.post_processing.export_vtk import export_modes_vtu

pvd_path = export_modes_vtu(
    nodes=nodes,
    elements=elements,
    modes_red=modes_red,
    freqs=freqs,
    idx_free=system["idx_free"],
    p0_nodes=system["p0_nodes"],
    output_dir="vtu_modes",
)
print(f"Open in ParaView: {pvd_path}")